In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef
from sklearn.metrics import confusion_matrix, classification_report

print("All libraries imported!")

All libraries imported!


In [2]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

print("Rows:", X.shape[0], "| Columns:", X.shape[1])
print("Classes:", data.target_names)

Rows: 569 | Columns: 30
Classes: ['malignant' 'benign']


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])

Train: 455 | Test: 114


In [4]:
# All 5 Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=5000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "kNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(random_state=42)
}

# Model Train and their Metrics
results = []

for name, model in models.items():
    # Train
    model.fit(X_train_scaled, y_train)

    # Predict
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    # 6 metrics
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred)
    })

print("All the models are trained")

All the models are trained


In [5]:
# Results in tabular form
results_df = pd.DataFrame(results)
results_df = results_df.round(4)   # 4 decimal tak

print("Comparison Table:")
results_df

Comparison Table:


,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9737,0.9974,0.9722,0.9859,0.9790,0.9439
1,Decision Tree,0.9474,0.9440,0.9577,0.9577,0.9577,0.8880
2,kNN,0.9474,0.9820,0.9577,0.9577,0.9577,0.8880
3,Naive Bayes,0.9649,0.9974,0.9589,0.9859,0.9722,0.9253
4,Random Forest,0.9649,0.9953,0.9589,0.9859,0.9722,0.9253


In [6]:
# Test data in a CSV file
test_data = X_test.copy()
test_data['target'] = y_test.values

test_data.to_csv('test_data.csv', index=False)

print("test_data.csv ban gaya!")
print("Rows:", test_data.shape[0], "| Columns:", test_data.shape[1])
test_data.head()

test_data.csv ban gaya!
Rows: 114 | Columns: 31


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
204,12.47,18.60,81.09,481.9,0.09965,0.1058,0.08005,0.03821,0.1925,0.06373,...,24.64,96.05,677.9,0.1426,0.2378,0.2671,0.10150,0.3014,0.08750,1
70,18.94,21.31,123.60,1130.0,0.09009,0.1029,0.10800,0.07951,0.1582,0.05461,...,26.58,165.90,1866.0,0.1193,0.2336,0.2687,0.17890,0.2551,0.06589,0
131,15.46,19.48,101.70,748.9,0.10920,0.1223,0.14660,0.08087,0.1931,0.05796,...,26.00,124.90,1156.0,0.1546,0.2394,0.3791,0.15140,0.2837,0.08019,0
431,12.40,17.68,81.47,467.8,0.10540,0.1316,0.07741,0.02799,0.1811,0.07102,...,22.91,89.61,515.8,0.1450,0.2629,0.2403,0.07370,0.2556,0.09359,1
540,11.54,14.44,74.65,402.9,0.09984,0.1120,0.06737,0.02594,0.1818,0.06782,...,19.68,78.78,457.8,0.1345,0.2118,0.1797,0.06918,0.2329,0.08134,1


In [7]:
import pickle
import os

# model folder
os.makedirs('model', exist_ok=True)

# Scaler save
with open('model/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# All models save
for name, model in models.items():
    filename = name.replace(" ", "_").lower()   # "Logistic Regression" -> "logistic_regression"
    with open(f'model/{filename}.pkl', 'wb') as f:
        pickle.dump(model, f)

print("All models are saved!")
print("Files:", os.listdir('model'))

Saare models save ho gaye!
Files: ['scaler.pkl', 'logistic_regression.pkl', 'random_forest.pkl', 'naive_bayes.pkl', 'decision_tree.pkl', 'knn.pkl']


In [8]:
app_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import pickle
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef,
                             confusion_matrix, classification_report)

# Page title
st.title("Breast Cancer Classification - Model Comparison")
st.write("Upload test data and compare 5 ML classification models.")

# Model names and their saved files
model_files = {
    "Logistic Regression": "logistic_regression.pkl",
    "Decision Tree": "decision_tree.pkl",
    "kNN": "knn.pkl",
    "Naive Bayes": "naive_bayes.pkl",
    "Random Forest": "random_forest.pkl"
}

# Feature 1: CSV upload
uploaded_file = st.file_uploader("Upload test_data.csv", type=["csv"])

if uploaded_file is not None:
    df = pd.read_csv(uploaded_file)
    st.subheader("Uploaded Data Preview")
    st.dataframe(df.head())

    # Separate features and target
    X_test = df.drop("target", axis=1)
    y_test = df["target"]

    # Load scaler and scale the data
    with open("model/scaler.pkl", "rb") as f:
        scaler = pickle.load(f)
    X_test_scaled = scaler.transform(X_test)

    # Feature 2: Model selection dropdown
    choice = st.selectbox("Select a model", list(model_files.keys()))

    # Load the chosen model
    with open("model/" + model_files[choice], "rb") as f:
        model = pickle.load(f)

    # Predict
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    # Feature 3: Show metrics
    st.subheader("Evaluation Metrics for " + choice)
    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }
    metrics_df = pd.DataFrame(metrics.items(), columns=["Metric", "Value"])
    metrics_df["Value"] = metrics_df["Value"].round(4)
    st.table(metrics_df)

    # Feature 4: Confusion matrix
    st.subheader("Confusion Matrix")
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots()
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    st.pyplot(fig)

    # Classification report
    st.subheader("Classification Report")
    report = classification_report(y_test, y_pred, output_dict=True)
    st.dataframe(pd.DataFrame(report).transpose().round(4))
else:
    st.info("Please upload the test_data.csv file to begin.")
'''

# save
with open('app.py', 'w') as f:
    f.write(app_code)

print("app.py is ready!")

app.py is ready!


In [9]:
requirements = '''streamlit
scikit-learn
pandas
numpy
matplotlib
seaborn
'''

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print("requirements.txt is ready!")
print(requirements)

requirements.txt is ready!
streamlit
scikit-learn
pandas
numpy
matplotlib
seaborn



In [10]:
import os
print("Files:", [f for f in os.listdir('.') if not f.startswith('.')])
print("Model folder:", os.listdir('model'))

Files: ['app.py', 'requirements.txt', 'model', 'test_data.csv', 'sample_data']
Model folder: ['scaler.pkl', 'logistic_regression.pkl', 'random_forest.pkl', 'naive_bayes.pkl', 'decision_tree.pkl', 'knn.pkl']


In [11]:
readme = '''# Breast Cancer Classification - ML Assignment 2

## a. Problem Statement
In this project, I have used machine learning to find out if a breast tumor is cancerous (malignant) or not (benign). Since there are only two possible answers, this is a binary classification problem. I have trained 5 different models and compared which one works best.

## b. Dataset Description
- **Dataset:** Breast Cancer Wisconsin Dataset (from UCI / scikit-learn)
- **Number of rows (instances):** 569
- **Number of features (columns):** 30 (like radius, texture, perimeter, area, etc.)
- **Target:** 0 = malignant, 1 = benign
- **Train/Test Split:** 80% for training, 20% for testing

## c. GitHub Repository Link
(ADD YOUR GITHUB LINK HERE)

## d. Models Used - Comparison Table

| ML Model | Accuracy | AUC | Precision | Recall | F1 | MCC |
|----------|----------|-----|-----------|--------|-----|-----|
| Logistic Regression | 0.9737 | 0.9974 | 0.9722 | 0.9859 | 0.9790 | 0.9439 |
| Decision Tree | 0.9474 | 0.9440 | 0.9577 | 0.9577 | 0.9577 | 0.8880 |
| kNN | 0.9474 | 0.9820 | 0.9577 | 0.9577 | 0.9577 | 0.8880 |
| Naive Bayes | 0.9649 | 0.9974 | 0.9589 | 0.9859 | 0.9722 | 0.9253 |
| Random Forest | 0.9649 | 0.9953 | 0.9589 | 0.9859 | 0.9722 | 0.9253 |

## Observations

| ML Model | Observation |
|----------|-------------|
| Logistic Regression | This gave the best results with the highest accuracy, F1 and MCC. The data can be separated with a straight line easily, so this simple model worked really well. |
| Decision Tree | Its AUC was the lowest (0.944). A single tree tends to memorize the training data too much, so it is not as strong. |
| kNN | It got the same accuracy as Decision Tree, but a better AUC (0.982). Scaling the data helped this model. |
| Naive Bayes | A simple model but it worked very well, with a very high AUC (0.997). |
| Random Forest | It gave good and stable results, similar to Naive Bayes. It is better than a single Decision Tree because it uses many trees together. |
| **Overall Winner** | **Logistic Regression** - it had the highest accuracy (0.9737), F1 (0.9790) and MCC (0.9439). |

## How to Run
1. Download the project files
2. Install the libraries: `pip install -r requirements.txt`
3. Start the app: `streamlit run app.py`
4. Upload the `test_data.csv` file in the app to see the results of each model.
'''

with open('README.md', 'w') as f:
    f.write(readme)

print("README.md is ready!")

README.md is ready!


In [12]:
import shutil
from google.colab import files

# important files in one folder
import os
os.makedirs('submission', exist_ok=True)

# Copy files into submission folder
shutil.copy('app.py', 'submission/app.py')
shutil.copy('requirements.txt', 'submission/requirements.txt')
shutil.copy('test_data.csv', 'submission/test_data.csv')
shutil.copy('README.md', 'submission/README.md')
shutil.copytree('model', 'submission/model', dirs_exist_ok=True)

# Zip
shutil.make_archive('submission', 'zip', 'submission')

# Download
files.download('submission.zip')

print("submission.zip is getting downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

submission.zip is getting downloaded!
